# EventPulse — College Event Feedback Analyzer

**Instantly understand how students felt about your college event — powered by Strands Decider 2B.**

This notebook walks through the full EventPulse pipeline:
1. Setup and configuration
2. Single feedback classification with Strands Decider 2B
3. Batch analysis across multiple responses
4. The reusable decision pattern
5. Benchmark results

**Tags:** `#strands-decider-2b` `#policy`  
**Repo:** https://github.com/Yugesh-003/eventpulse  
**Built with the AWS User Group Madurai Strands Decider hackathon kit**

## 1. Install dependencies

Clone the repo and install requirements. Skip if already installed.

In [ ]:
# Clone the repo (skip if already cloned)
import os
if not os.path.exists('eventpulse'):
    !git clone https://github.com/Yugesh-003/eventpulse.git
os.chdir('eventpulse')
!pip install -r requirements.txt -q

## 2. Configure the Decider

EventPulse uses Strands Decider 2B via the AWS User Group Madurai hosted API.  
Set your API key below (get yours at https://www.awsugmdu.in).

In [ ]:
import os

# Set your AWS User Group Madurai decider API key
os.environ['DECIDER_MODE'] = 'ugmdu'
os.environ['DECIDER_URL'] = 'https://decider.awsugmdu.in'
os.environ['DECIDER_API_KEY'] = 'YOUR_KEY_HERE'  # replace with your key

# LLM provider (ugmdu = user group cloud, bedrock = Amazon Bedrock)
os.environ['LLM_PROVIDER'] = 'ugmdu'
os.environ['LLM_MODEL_ID'] = 'qwen3.5-4b'
os.environ['AWS_REGION'] = 'us-east-1'

print('Environment configured.')

## 3. The decision pattern

The core of EventPulse is a single `choice` question sent to Strands Decider 2B.  
The model reads the feedback text and returns a probability distribution over three labels.

In [ ]:
from decider_kit import Decider

# The typed question handed to Strands Decider 2B
QUESTIONS = {
    "sentiment": Decider.choice(
        "What is the overall sentiment expressed in this college event feedback?",
        {
            "positive": "the student is satisfied, happy, enthusiastic, or praises the event or its organisation",
            "negative": "the student is dissatisfied, disappointed, frustrated, or criticises the event or its organisation",
            "neutral":  "the student neither praises nor criticises clearly; the response is factual, indifferent, or mixed without a clear lean",
        },
    )
}

CONFIDENCE_THRESHOLD = 0.7

print('Decision pattern loaded.')

## 4. Single feedback classification

Let's classify a single feedback response directly using Strands Decider 2B.

In [ ]:
from agent.decision import FeedbackSentimentDecider

d = Decider()
clf = FeedbackSentimentDecider(d)

# Try a positive feedback
result = clf.classify(
    "The hackathon was absolutely incredible! I learned so much and the mentors were super helpful.",
    event_name="Tech Fest 2024"
)

print(f"Verdict:    {result['verdict']}")
print(f"Label:      {result['label']}")
print(f"Confidence: {result['confidence']}")
print(f"Decider ms: {result['decider_ms']}")

In [ ]:
# Try a negative feedback
result = clf.classify(
    "The food stalls ran out of food by noon. People were waiting in long queues and the staff was rude. Terrible experience.",
    event_name="Tech Fest 2024"
)
print(f"Verdict: {result['verdict']}  |  Confidence: {result['confidence']}  |  {result['decider_ms']} ms")

In [ ]:
# Try a neutral feedback (near the threshold — may be flagged for review)
result = clf.classify(
    "The event happened as scheduled. There were workshops in the morning and a competition in the afternoon.",
    event_name="Tech Fest 2024"
)
print(f"Verdict: {result['verdict']}  |  Confidence: {result['confidence']}  |  {result['decider_ms']} ms")
if result['verdict'] == 'needs_review':
    print("=> Confidence below threshold. Flagged for human review.")

## 5. Batch analysis

Classify a batch of feedback responses and get a summary breakdown.

In [ ]:
import json
from agent.tools import batch_analyze

feedback_batch = [
    "Amazing workshops, very well organized. The AR/VR demo was mind-blowing!",
    "Very disappointing. The event started 2 hours late and there was no shade for spectators.",
    "The seminar covered machine learning basics and some applications. It lasted about two hours.",
    "Loved the coding competition. The prizes were generous and the judging was fair. Truly inspiring!",
    "The main performance was outstanding but the opening act was painful to sit through. Mixed feelings.",
]

result = batch_analyze(json.dumps(feedback_batch), event_name="Various Events")

print(f"Total responses: {result['total']}")
print(f"Summary: {result['summary']}")
if result['flagged_for_review']:
    print(f"Flagged for review (indices): {result['flagged_for_review']}")
print()
for r in result['results']:
    flag = ' [REVIEW]' if r['verdict'] == 'needs_review' else ''
    print(f"  [{r['verdict'].upper():<12}] conf={r['confidence']:.3f}{flag}  {r['feedback'][:60]}...")

## 6. Full agent conversation

Use the full `HackathonAgent` with Amazon Bedrock (or the user group LLM) for the conversational layer.

In [ ]:
from agent.agent import HackathonAgent

agent = HackathonAgent()

# Turn 1: single analysis
out = agent.chat("Analyze this feedback from our Cultural Night: The performances were breathtaking. Truly professional.")
print("Reply:", out['reply'])
print("Decisions:", out['decisions'])
print(f"Latency: {out['latency_ms']} ms")

In [ ]:
# Turn 2: batch analysis via chat
out = agent.chat(
    'Can you batch analyze these: ["Amazing event!", "Terrible organization.", "The event lasted 3 hours."]',
    session_id="default"
)
print("Reply:", out['reply'])

## 7. Benchmark results

These results come from `submission/bench/results.json` — a real run of `./kit bench --split test`.

In [ ]:
import json
from pathlib import Path

results = json.loads(Path('submission/bench/results.json').read_text())
s = results['summary']
decider = s['decider']
llm = s['llm']

print(f"Decider model : {results['decider_model']}")
print(f"Test cases    : {results['cases']}")
print()
print(f"{'Metric':<35} {'Strands Decider 2B':>20} {'LLM baseline':>15}")
print("-" * 72)
print(f"{'Accuracy':<35} {decider['accuracy']:>19.1%} {llm['accuracy']:>14.1%}")
print(f"{'Median latency (ms)':<35} {decider['latency_p50_ms']:>20.0f} {llm['latency_p50_ms']:>15.0f}")
print(f"{'p95 latency (ms)':<35} {decider['latency_p95_ms']:>20.0f} {llm['latency_p95_ms']:>15.0f}")
print(f"{'Autonomous coverage @ 0.7':<35} {decider['confident_share']:>19.1%} {llm['confident_share']:>14.1%}")
print(f"{'Accuracy when confident':<35} {decider['accuracy_when_confident']:>19.1%} {llm['accuracy_when_confident']:>14.1%}")
print()
print(f"Speedup: {s['speedup_p50']:.1f}x faster at the median")

In [ ]:
# Plot latency comparison
import matplotlib.pyplot as plt

models = ['Strands Decider 2B', 'LLM baseline']
latencies = [decider['latency_p50_ms'], llm['latency_p50_ms']]
colors = ['#FF9900', '#232F3E']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(models, latencies, color=colors, width=0.4)
ax.set_ylabel('Median latency (ms)')
ax.set_title('Same Accuracy, Nearly 7x Faster Decisions')
for bar, val in zip(bars, latencies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{val:.0f} ms', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, max(latencies) * 1.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 8. How to reuse the decision pattern

The pattern in `agent/decision.py` is designed to be dropped into any agent.

```python
from decider_kit import Decider
from agent.decision import FeedbackSentimentDecider

clf = FeedbackSentimentDecider(Decider())
result = clf.classify("Your feedback text here", event_name="Your Event")

if result['verdict'] == 'needs_review':
    # route to human
    pass
else:
    # act on result['verdict']: 'positive', 'negative', or 'neutral'
    print(result['verdict'], result['confidence'])
```

To adapt it to a different domain, change the three options in `QUESTIONS`:  
- Support tickets: `satisfied / unsatisfied / escalate`  
- Content moderation: `relevant / irrelevant / spam`  
- Product reviews: `recommend / neutral / do_not_recommend`

---

**Code:** https://github.com/Yugesh-003/eventpulse  
**Tags:** #strands-decider-2b #policy  
**Built with the AWS User Group Madurai Strands Decider hackathon kit**